# HAKE-MER — Step~0 cardinality (DistilBERT PLM baseline)

Fills the **Step~0** column of `tab:expe-cardinality` without changing canonical campaign F1 scores.

1. Ensures three `runs/distilbert_base_uncased_seed*_baseline_plm/best_model.pt` exist (trains missing seeds with the usual Step~0 protocol).
2. Runs `./run_step0_cardinality.sh` (test-only `by_gold_cardinality`).
3. Download zip → integrate into `step_eval_supplements_distilbert/`.

Reference macro: **Step~0 F1-macro test ≈ 0.486 ± 0.005** (canonical campaign JSON).

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
3807903


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.3 MB/s eta 0:00:00


In [4]:
from pathlib import Path

ck = sorted(Path("runs").glob("distilbert_base_uncased_seed*_baseline_plm/best_model.pt"))
print(len(ck), "/ 3 Step~0 checkpoints")
if len(ck) < 3:
    print("Training Step~0 PLM (missing seeds only in campaign runner)…")
    get_ipython().system(f"./run_baseline_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}")

0 / 3 Step~0 checkpoints
Training Step~0 PLM (missing seeds only in campaign runner)…
config.json: 100% 483/483 [00:00<00:00, 5.13MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 654kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 9.69MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 33.3MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 30.5MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:   4% 123k/2.77M [00:00<00:06, 397kB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 5.66MB/s,  269kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 5.74MB/s,  273kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 1.99MB/s,   ???B/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:00<00:00, 2.01MB/s,   

In [5]:
!chmod +x run_step0_cardinality.sh && ALLOW_EVAL=1 DEVICE=cuda ./run_step0_cardinality.sh

No supplements applied (missing checkpoints or already complete).
seed 42: cardinality from distilbert_base_uncased_seed42_baseline_plm
seed 123: cardinality from distilbert_base_uncased_seed123_baseline_plm
seed 456: cardinality from distilbert_base_uncased_seed456_baseline_plm
Wrote /content/marii/reference/artifacts/cardinality_snippet.tex
1 & $0{,}502 \pm 0{,}007$ & $0{,}490 \pm 0{,}007$ & $0{,}506 \pm 0{,}003$ \\
\hline
2 & $0{,}437 \pm 0{,}018$ & $0{,}361 \pm 0{,}003$ & $0{,}488 \pm 0{,}009$ \\
\hline
3 & $0{,}315 \pm 0{,}018$ & $0{,}288 \pm 0{,}037$ & $0{,}320 \pm 0{,}018$ \\
\hline
4+ & $0{,}052 \pm 0{,}006$ & $0{,}032 \pm 0{,}011$ & $0{,}048 \pm 0{,}000$ \\
\hline
Updated supplement baseline campaign and cardinality_snippet.tex


In [6]:
from pathlib import Path
print(Path("reference/artifacts/cardinality_snippet.tex").read_text())

1 & $0{,}502 \pm 0{,}007$ & $0{,}490 \pm 0{,}007$ & $0{,}506 \pm 0{,}003$ \\
\hline
2 & $0{,}437 \pm 0{,}018$ & $0{,}361 \pm 0{,}003$ & $0{,}488 \pm 0{,}009$ \\
\hline
3 & $0{,}315 \pm 0{,}018$ & $0{,}288 \pm 0{,}037$ & $0{,}320 \pm 0{,}018$ \\
\hline
4+ & $0{,}052 \pm 0{,}006$ & $0{,}032 \pm 0{,}011$ & $0{,}048 \pm 0{,}000$ \\
\hline



In [7]:
import zipfile
from google.colab import files
from pathlib import Path

out_name = "step0_cardinality_distilbert.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    bc = Path("reference/artifacts/step_eval_supplements_distilbert/campaigns/baseline_plm_distilbert_base_uncased_campaign.json")
    zf.write(bc, bc.name)
    zf.write(Path("reference/artifacts/cardinality_snippet.tex"), "cardinality_snippet.tex")
    for m in sorted(Path("runs").glob("distilbert_base_uncased_seed*_baseline_plm/metrics.json")):
        zf.write(m, f"metrics/{m.parent.name}/{m.name}")
files.download(str(zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Send **zip** + executed **.ipynb** for repo integration.